# Leukemia Detection Model Training on Google Colab (.h5 Model Output)

This notebook trains a **ResNet50 Multiclass Classifier** on your 5-class Leukemia Dataset (`ALL`, `AML`, `CLL`, `CML`, `Normal`) using GPU acceleration in Google Colab and exports the trained model directly as a **`.h5`** file.

### Classes Mapped:
- **ALL**: `ALL TEST-20230225T082325Z-001` (Acute Lymphoblastic Leukemia)
- **AML**: `AML TEST-20230225T082630Z-001` (Acute Myeloid Leukemia)
- **CLL**: `CLL TEST-20230225T082851Z-001` (Chronic Lymphocytic Leukemia)
- **CML**: `CML TEST-20230225T083148Z-001` (Chronic Myeloid Leukemia)
- **Normal**: `H TEST-20230225T083612Z-001` (Normal Blood Smear)

## Step 1: Check GPU Acceleration

In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU detected:", gpus[0].name)
else:
    print("⚠️ No GPU detected. Please go to Runtime -> Change runtime type -> Hardware Accelerator -> GPU.")

## Step 2: Upload or Mount `dataset.zip`

Option A: Mount Google Drive (Recommended for large datasets, e.g., 3.6 GB)
Option B: Direct upload via Google Colab sidebar

In [ ]:
# Mount Google Drive if dataset.zip is stored on Google Drive
from google.colab import drive
import os
import zipfile

drive.mount('/content/drive')

# Update path if your dataset.zip is in Google Drive or root Colab environment
zip_path = '/content/drive/MyDrive/dataset.zip' # Or '/content/dataset.zip'
extract_path = '/content/dataset_raw'

if not os.path.exists(zip_path) and os.path.exists('/content/dataset.zip'):
    zip_path = '/content/dataset.zip'

print(f"Unzipping dataset from {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("✅ Dataset extraction completed!")

## Step 3: Organize Dataset into Standard Class Directory Structure

In [ ]:
import shutil
from glob import glob

dataset_dir = '/content/dataset'
os.makedirs(dataset_dir, exist_ok=True)

class_folder_map = {
    "ALL": "ALL TEST-20230225T082325Z-001",
    "AML": "AML TEST-20230225T082630Z-001",
    "CLL": "CLL TEST-20230225T082851Z-001",
    "CML": "CML TEST-20230225T083148Z-001",
    "Normal": "H TEST-20230225T083612Z-001"
}

for cls_name, subfolder in class_folder_map.items():
    target_dir = os.path.join(dataset_dir, cls_name)
    os.makedirs(target_dir, exist_ok=True)
    
    # Find all images inside the unzipped dataset
    search_pattern = os.path.join(extract_path, "**", subfolder, "**", "*.[jJ][pP]*[gG]")
    images = glob(search_pattern, recursive=True)
    
    if not images:
        # Fallback search if path structure varies
        search_pattern_alt = os.path.join(extract_path, "**", subfolder, "*.[jJ][pP]*[gG]")
        images = glob(search_pattern_alt, recursive=True)
        
    print(f"Class '{cls_name}': Found {len(images)} images.")
    
    for img_path in images:
        filename = os.path.basename(img_path)
        dest_path = os.path.join(target_dir, filename)
        if not os.path.exists(dest_path):
            shutil.copy(img_path, dest_path)

print("\n✅ Organized Dataset Structure:")
for cls_name in class_folder_map.keys():
    c_count = len(os.listdir(os.path.join(dataset_dir, cls_name)))
    print(f" - {cls_name}: {c_count} images")

## Step 4: Data Augmentation & Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("Class Indices:", train_generator.class_indices)

## Step 5: Build ResNet50 Classifier Model

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, optimizers

# Load pretrained ResNet50 without top classification layers
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Unfreeze top layers for fine-tuning
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
predictions = layers.Dense(5, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

model.summary()

## Step 6: Train Model & Save `.h5` File

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Path where the .h5 model will be saved
h5_model_filename = 'leukemia_resnet50.h5'

callbacks = [
    ModelCheckpoint(
        h5_model_filename,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

EPOCHS = 15

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks
)

# Ensure model is explicitly saved to .h5
model.save('leukemia_resnet50_final.h5')
print(f"\n🎉 Training Complete! Model saved successfully as '{h5_model_filename}' and 'leukemia_resnet50_final.h5'!")

## Step 7: Download the Trained `.h5` Model File

In [ ]:
from google.colab import files

print("Downloading trained .h5 model to your computer...")
files.download('leukemia_resnet50.h5')